# Worker A: Healthcare (Full Scale)

Runs the full Obermeyer healthcare grid. Uses analytic (KKT-based) decision gradients.

**Grid:** 7 methods × 2 alphas × 5 seeds = 70 runs

Checkpointed: re-run cells to resume from where you left off.

In [ ]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# Results directory
HC_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'healthcare_v2')
os.makedirs(HC_RESULTS, exist_ok=True)

from experiments.colab_runner import *
import torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Results: {HC_RESULTS}')

Mounted at /content/drive
MOSEK license set: /content/drive/MyDrive/DecisionFocusedMTL/data/mosek.lic
/content/drive/MyDrive/DecisionFocusedMTL
Device: cuda
Results: /content/drive/MyDrive/DecisionFocusedMTL/results/final/healthcare_v2


## Configuration

In [ ]:
# ===== TASK PARAMETERS =====
TASK_OVERRIDES = {
    'n_sample': 0,                 # 0 = all 48,784 patients (full scale)
    'val_fraction': 0.1,           # 10% validation split
    # 'budget_rho': 0.35,
    # 'test_fraction': 0.5,
    # 'decision_mode': 'group',
    # 'fairness_type': 'mad',
}

# ===== TRAINING PARAMETERS =====
TRAIN_OVERRIDES = {
    # Optimizer
    'optimizer': 'adamw',           # 'sgd', 'sgd_momentum', 'adam', 'adamw'
    'lr': 0.001,                    # Learning rate
    'weight_decay': 1e-4,           # Decoupled weight decay (adamw)
    'lr_warmup_steps': 5,           # Linear LR warmup
    # Model architecture
    'init_mode': 'best_practice',   # Kaiming He initialization
    'dropout': 0.1,                 # Dropout rate
    'hidden_dim': 64,               # MLP hidden width
    'n_layers': 2,                  # MLP depth
}

# ===== EXPERIMENT CONTROL =====
STEPS = 70
OVERWRITE = False

print('='*60)
print('HEALTHCARE EXPERIMENT — FULL SCALE')
print('='*60)
print(f'Steps per lambda: {STEPS}')
print(f'Overwrite: {OVERWRITE}')
print(f'Task overrides: {TASK_OVERRIDES}')
print(f'Train overrides: {TRAIN_OVERRIDES}')
print(f'Results dir: {HC_RESULTS}')
print('='*60)

HEALTHCARE EXPERIMENT — FULL SCALE
Steps per lambda: 70
Overwrite: False
Task overrides: {'n_sample': 0, 'val_fraction': 0.1}
Train overrides: {'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.0001, 'lr_warmup_steps': 5, 'init_mode': 'best_practice', 'dropout': 0.1, 'hidden_dim': 64, 'n_layers': 2}
Results dir: /content/drive/MyDrive/DecisionFocusedMTL/results/final/healthcare_v2


## Run All Methods

In [3]:
results = run_healthcare_slice(
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
    overwrite=OVERWRITE,
)


Healthcare slice: ['FPTO', 'SAA', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
Alphas=[0.5, 2.0], HiddenDims=[64], Seeds=[11, 22, 33, 44, 55]
Device=cuda, Total runs=70
Task overrides: {'n_sample': 0, 'val_fraction': 0.1}
Train overrides: {'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.0001, 'lr_warmup_steps': 5, 'init_mode': 'best_practice', 'dropout': 0.1, 'hidden_dim': 64, 'n_layers': 2}
  [1/70] FPTO a=0.5 hd=64 s=11 (23.2s)
    Est. remaining: 27min
  [2/70] FPTO a=0.5 hd=64 s=22 (15.1s)
  [3/70] FPTO a=0.5 hd=64 s=33 (15.4s)
  [4/70] FPTO a=0.5 hd=64 s=44 (15.3s)
  [5/70] FPTO a=0.5 hd=64 s=55 (14.8s)
  [6/70] FPTO a=2.0 hd=64 s=11 (14.9s)
  [7/70] FPTO a=2.0 hd=64 s=22 (14.8s)
  [8/70] FPTO a=2.0 hd=64 s=33 (14.7s)
  [9/70] FPTO a=2.0 hd=64 s=44 (14.9s)
  [10/70] FPTO a=2.0 hd=64 s=55 (14.7s)
  [11/70] SAA a=0.5 hd=64 s=11 (1.1s)
  [12/70] SAA a=0.5 hd=64 s=22 (1.2s)
  [13/70] SAA a=0.5 hd=64 s=33 (0.9s)
  [14/70] SAA a=0.5 hd=64 s=44 (0.8s)
  [15/70] S

## Results

In [4]:
show_progress(HC_RESULTS, 'Healthcare — COMPLETE')

if not results.empty:
    cols = ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
    for alpha in sorted(results['alpha_fair'].unique()):
        sub = results[results['alpha_fair'] == alpha]
        print(f'\n{"="*60}')
        print(f'alpha = {alpha} (mean +/- std over seeds)')
        print(f'{"="*60}')
        summary = sub.groupby(['method_label', 'lambda'])[cols].agg(['mean', 'std']).round(4)
        print(summary.to_string())

        # SAA comparison
        saa = sub[sub['method_label'] == 'SAA']
        if not saa.empty:
            saa_reg = saa['test_regret_normalized'].mean()
            saa_mse = saa['test_pred_mse'].mean()
            print(f'\nSAA baseline: Regret={saa_reg:.4f}, MSE={saa_mse:.1f}')
            for m in ['FPTO', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']:
                ms = sub[(sub['method_label'] == m) & (sub['lambda'] == 0.0)]
                if not ms.empty:
                    r = ms['test_regret_normalized'].mean()
                    print(f'  {m:15s}: Regret={r:.4f} ({(r-saa_reg)/saa_reg*100:+.1f}% vs SAA)')


Healthcare — COMPLETE — 130 rows from 70 runs
Methods: ['FDFL-CAGrad', 'FDFL-MGDA', 'FDFL-PCGrad', 'FDFL-Scal', 'FPTO', 'SAA', 'WDRO']
Seeds:   [np.int64(11), np.int64(22), np.int64(33), np.int64(44), np.int64(55)]
Alphas:  [np.float64(0.5), np.float64(2.0)]

--- alpha = 0.5 (mean +/- std over seeds) ---
                     test_regret_normalized_mean  test_regret_normalized_std  test_fairness_mean  test_fairness_std  test_pred_mse_mean  test_pred_mse_std
method_label lambda                                                                                                                                       
FDFL-CAGrad  0.0000                       0.0557                      0.0008             23.0954             0.6404            170.9949             3.6725
FDFL-MGDA    0.0000                       0.0549                      0.0009             69.2989             6.4740            323.1522            16.0074
FDFL-PCGrad  0.0000                       0.0544                      0.0

Worker A complete. Run the Results notebook to analyze.